ARTI308 - Machine Learning

# Linear Regression on House Rent Dataset


**Let's get started!**  
## Check out the data

In this notebook, we will follow the **same workflow** used in the lab notebook, but we will apply it to a **new dataset**: **House Rent Dataset**.

Our goal is to predict **Rent** using house-related features.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

%matplotlib inline

### Check out the Data

In [ ]:
HouseRent = pd.read_csv('House_Rent_Dataset.csv')

In [ ]:
HouseRent.head()

In [ ]:
HouseRent.info()

In [ ]:
HouseRent.describe()

In [ ]:
HouseRent.columns

# EDA

Let's create some simple plots to understand the dataset before modeling.


In [ ]:
sns.pairplot(HouseRent[['BHK','Rent','Size','Bathroom']])

In [ ]:
sns.histplot(HouseRent['Rent'], bins=50, kde=True)

In [ ]:
numeric_df = HouseRent[['BHK','Rent','Size','Bathroom']]
sns.heatmap(numeric_df.corr(), annot=True, cmap='viridis')

## Basic Data Cleaning and Feature Engineering

Unlike the original lab dataset, this dataset contains several **categorical** columns and a **Floor** column written as text such as:

- `Ground out of 2`
- `3 out of 5`
- `Upper Basement out of 4`

So we need to do a little preprocessing first.


In [ ]:
HouseRent.isnull().sum()

In [ ]:
HouseRent['Posted On'] = pd.to_datetime(HouseRent['Posted On'])

HouseRent['Posted_Month'] = HouseRent['Posted On'].dt.month
HouseRent['Posted_Day'] = HouseRent['Posted On'].dt.day

def parse_floor(value):
    value = str(value).strip()

    if ' out of ' in value:
        current, total = value.split(' out of ', 1)
    else:
        current, total = value, value

    current_lower = current.lower().strip()

    if current_lower == 'ground':
        current_floor = 0
    elif 'upper basement' in current_lower:
        current_floor = -1
    elif 'lower basement' in current_lower:
        current_floor = -2
    else:
        nums = re.findall(r'-?\d+', current)
        current_floor = int(nums[0]) if nums else np.nan

    total_nums = re.findall(r'\d+', total)
    total_floors = int(total_nums[0]) if total_nums else np.nan

    return pd.Series([current_floor, total_floors])

HouseRent[['Current_Floor', 'Total_Floors']] = HouseRent['Floor'].apply(parse_floor)
HouseRent['Total_Floors'] = HouseRent['Total_Floors'].fillna(HouseRent['Current_Floor']).fillna(0)

HouseRent.head()

## Training a Linear Regression Model

Now let's prepare our **X** features and **y** target.

We will predict:

- **y = Rent**

We will use:
- numeric features like `BHK`, `Size`, `Bathroom`
- engineered features like `Posted_Month`, `Posted_Day`, `Current_Floor`, `Total_Floors`
- categorical features converted into dummy variables


In [ ]:
X = HouseRent[['BHK',
               'Size',
               'Bathroom',
               'Posted_Month',
               'Posted_Day',
               'Current_Floor',
               'Total_Floors',
               'Area Type',
               'City',
               'Furnishing Status',
               'Tenant Preferred',
               'Point of Contact']]

y = HouseRent['Rent']

In [ ]:
X = pd.get_dummies(X, drop_first=True)
X.head()

## Train Test Split

Now let's split the data into a training set and a testing set.


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=101
)

## Creating and Training the Model

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lm = LinearRegression()

In [ ]:
lm.fit(X_train, y_train)

## Model Evaluation

Let's check the intercept and coefficients.


In [ ]:
print(lm.intercept_)

In [ ]:
coeff_df = pd.DataFrame(lm.coef_, X.columns, columns=['Coefficient'])
coeff_df.sort_values('Coefficient', ascending=False)

### Quick interpretation

Each coefficient tells us how the predicted **Rent** changes when that feature increases by **1 unit**, while all other features stay fixed.

For dummy variables:
- a **positive** coefficient means that category tends to increase the predicted rent compared with the baseline category
- a **negative** coefficient means that category tends to decrease the predicted rent compared with the baseline category


## Predictions from our Model

Let's make predictions using the test data.


In [ ]:
predictions = lm.predict(X_test)

In [ ]:
plt.scatter(y_test, predictions)
plt.xlabel('Actual Rent')
plt.ylabel('Predicted Rent')

**Residual Histogram**

In [ ]:
sns.histplot((y_test - predictions), bins=50, kde=True)

## Regression Evaluation Metrics

In [ ]:
from sklearn import metrics

In [ ]:
print('R2 Score:', lm.score(X_test, y_test))
print('MAE:', metrics.mean_absolute_error(y_test, predictions))
print('MSE:', metrics.mean_squared_error(y_test, predictions))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_test, predictions)))

This notebook applies the **same lab steps** on a new dataset.

### Task Completed:
1. Loaded the dataset into a DataFrame  
2. Explored the data (`head`, `info`, `describe`)  
3. Performed basic data cleaning  
4. Applied feature engineering  
5. Prepared the data for modeling  
6. Trained a **Linear Regression** model  
7. Evaluated the model performance  

### Notes
- The dataset contains categorical columns, so we used **dummy variables**
- The `Floor` column was converted from text into numeric floor features
- This notebook is ready to be used in **Jupyter Notebook**
